# Viper QLoRA fine-tune (M4, gated)

**Run only after the M4 gate re-opens** (`docs/plans/m4-qlora-gate.md`): a repeatable model-behavior failure must survive the cheap prompt/retrieval fixes first. Current decision: *do not train*.

Rules enforced here:
- Base model **Foundation-Sec-8B** (fdtn-ai). **Verify the Llama 3.1 Community License** before training (cell below).
- Training data comes **only** from `train.jsonl` built by `viper.training.dataset`, which excludes the held-out eval records. Never add the eval datasets.
- The adapter is accepted **only** if it beats the frozen v2 baseline with no general/safety regression, and a human approves. Keep the previous model for rollback.

In [ ]:
# 1. Environment (Colab GPU). Pin versions when you run for real.
!pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.9" "bitsandbytes>=0.43" "datasets>=2.20" accelerate
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 2. License check — read and confirm the Llama 3.1 Community License + Cisco Apache-2.0 NOTICE
#    for the chosen Foundation-Sec-8B variant BEFORE downloading weights.
BASE_MODEL = 'fdtn-ai/Foundation-Sec-8B'  # base (Apache-2.0 card, derived from Llama-3.1-8B)
print('Confirm license terms for', BASE_MODEL, 'then set LICENSE_OK = True')
LICENSE_OK = False
assert LICENSE_OK, 'Verify the model license before training.'

In [ ]:
# 3. Load train.jsonl (built locally by: python -m viper.training build-dataset).
#    Re-assert disjointness from the held-out eval set here too (defense in depth).
from datasets import load_dataset
ds = load_dataset('json', data_files='train.jsonl', split='train')
def fmt(r):
    return {'text': f"<|user|>\n{r['instruction']}\n<|assistant|>\n{r['output']}"}
ds = ds.map(fmt)
print(len(ds), 'examples; sample:', ds[0]['text'][:200])

In [ ]:
# 4. QLoRA: 4-bit base + LoRA adapters, short SFT run. Tune hyperparameters per the gate.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(BASE_MODEL); tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj'])
cfg = SFTConfig(output_dir='adapter', num_train_epochs=1, per_device_train_batch_size=1,
                gradient_accumulation_steps=8, learning_rate=2e-4, bf16=True, logging_steps=10,
                dataset_text_field='text', max_seq_length=2048)
SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora).train()
model.save_pretrained('adapter'); print('saved adapter/')

## 5. Export & evaluate (back on the workstation / Colab)
1. Download `adapter/` and run `viper.training.artifact.verify_adapter(adapter/)`.
2. Merge the adapter into the base and export GGUF (llama.cpp `convert` + `quantize`).
3. `render_modelfile(...)` -> `ollama create viper-sec-cand -f Modelfile`.
4. Run the **full frozen v2 answer eval** on this candidate and `viper eval compare baseline.json candidate.json`.
5. Accept only on a win with no general/safety regression **and** human approval; keep the previous model for rollback.